
# SURFMATCH — Nylon planar projection (corrected reference pipeline)

Questo notebook costruisce **solo la proiezione planare del nylon**.

Non estrae ROI e non confronta ancora la proiezione con SURFMAP: quel confronto va fatto in un notebook separato.

## Output principali

Per ogni punto selezionato della superficie frontale conserva:

- `point_id`
- coordinate originali `x, y, z`
- coordinate metriche sul piano `u, v` in Å
- `radial_distance_A` dal centro geometrico della faccia
- `depth_A` rispetto al piano di primo contatto
- `signed_contact_distance_A`
- `charge_raw`
- `charge_normalized`

Produce inoltre:

1. controllo 3D della geometria;
2. mappa 2D della **depth**;
3. mappa 2D delle **cariche** sugli stessi identici punti;
4. CSV dei punti proiettati;
5. JSON con i metadati geometrici;
6. HTML interattivi delle mappe.

## Concetti separati

- **Orientation plane**: definisce come è orientata la faccia.
- **Contact plane**: piano parallelo posto al punto più esterno; qui `depth = 0`.
- **Depth cutoff**: definisce quanto spessore frontale considerare.
- **Face center**: definisce `(u,v) = (0,0)`.
- **u,v**: coordinate fisiche in Å sul piano.
- **depth**: distanza verso l'interno dal contact plane.
- **charge**: proprietà elettrostatica dello stesso punto.

> Nessuna ROI viene applicata in questo notebook.



## 1. Dipendenze

Da terminale, nell'environment usato per il notebook:

```bash
pip install numpy pandas plotly scikit-spatial scipy
```

PyMOL non è necessario.


In [1]:

from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from scipy.spatial import cKDTree
from skspatial.objects import Plane

print("Libraries loaded.")


Libraries loaded.



## 2. Parameters

Deterministic outward normal

A fitted plane has two mathematically equivalent normal vectors, n and -n. However, SURFMATCH requires a consistent physical direction because depth must increase toward the interior of the material. The normal is therefore oriented so that the bulk of the nylon lies behind the plane, while the positive normal points outward.

In [2]:

# ============================================================
# INPUT FILES
# ============================================================

NEUTRAL_SHELL_PATH = Path(
    "/home/chiara/Desktop/COMENZE/Comenze_pipeline/"
    "Nylon_surface/Nylon_surfmap/Nyl_rad0.5_s2_nosmooth/"
    "output_SURFMAP_NYL_newVer_rotY90resnum_bfactor/"
    "NYL_newVer_rotY90resnum_bfactor_mapped_shell.pdb"
)

# Inserire qui il file che contiene la carica per gli stessi punti dello shell.
# Può anche essere lo stesso PDB, se la carica è già salvata nel B-factor.
CHARGED_SHELL_PATH = "/home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_surfmap/Nyl_rad0.5_s2_nosmooth/output_SURFMAP_NYL_newVer_rotY90resnum_bfactor/NYL_newVer_rotY90resnum_bfactor_mapped_shell.pdb"
# Esempio:
# CHARGED_SHELL_PATH = Path("/home/chiara/.../charged_shell.pdb")

# "bfactor", "occupancy", "pqr", "csv"
CHARGE_SOURCE = "bfactor"

# ============================================================
# ORIGINAL REFERENCE POINTS
# ============================================================

REFERENCE_POINTS = np.array(
    [
        [28.841, 65.113, 58.105],
        [15.231, 30.442, 49.824],
        [3.007, 4.288, 41.563],
        [-2.145, 65.211, 39.752],
        [31.409, 1.629, 58.441],
    ],
    dtype=float,
)

# ============================================================
# FRONT SURFACE / DEPTH
# ============================================================

FACE_DEPTH_CUTOFF_A = 8.0

# 1.0 = vero massimo geometrico / piano di primo contatto.
CONTACT_QUANTILE = 1.0

# ============================================================
# OPTIONAL ORIENTATION REFIT
# ============================================================

REFIT_ORIENTATION = False
REFIT_BAND_A = 1.0

# ============================================================
# LOCAL 2D COORDINATE SYSTEM
# ============================================================

PREFERRED_U_AXIS = np.array([1.0, 0.0, 0.0])

# ============================================================
# CHARGE MATCHING
# ============================================================

COORD_MATCH_TOL_A = 0.002

# ============================================================
# VISUALIZATION
# ============================================================

PLANE_HALF_SIZE_A = 45.0
MARKER_SIZE = 5

print("Neutral shell :", NEUTRAL_SHELL_PATH)
print("Charged shell :", CHARGED_SHELL_PATH)
print("Depth cutoff  :", FACE_DEPTH_CUTOFF_A, "Å")


Neutral shell : /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_surfmap/Nyl_rad0.5_s2_nosmooth/output_SURFMAP_NYL_newVer_rotY90resnum_bfactor/NYL_newVer_rotY90resnum_bfactor_mapped_shell.pdb
Charged shell : /home/chiara/Desktop/COMENZE/Comenze_pipeline/Nylon_surface/Nylon_surfmap/Nyl_rad0.5_s2_nosmooth/output_SURFMAP_NYL_newVer_rotY90resnum_bfactor/NYL_newVer_rotY90resnum_bfactor_mapped_shell.pdb
Depth cutoff  : 8.0 Å



## 3. Reading files

La geometria e la carica vengono mantenute separate.

Il file geometrico fornisce i punti dello shell. Il file caricato deve descrivere gli **stessi punti**; il notebook verifica la corrispondenza geometrica prima di trasferire i valori di carica.


In [3]:

def read_pdb(path: Path) -> pd.DataFrame:
    rows = []
    atom_index = 0

    with path.open("r") as handle:
        for line in handle:
            if not line.startswith(("ATOM", "HETATM")):
                continue

            try:
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])

                occ_txt = line[54:60].strip()
                bfac_txt = line[60:66].strip()

                occupancy = float(occ_txt) if occ_txt else np.nan
                bfactor = float(bfac_txt) if bfac_txt else np.nan

                rows.append(
                    (atom_index, x, y, z, occupancy, bfactor)
                )
                atom_index += 1

            except ValueError:
                continue

    df = pd.DataFrame(
        rows,
        columns=[
            "file_index", "x", "y", "z",
            "occupancy", "bfactor"
        ],
    )

    if df.empty:
        raise ValueError(f"No ATOM/HETATM coordinates found in:\n{path}")

    return df


def read_pqr(path: Path) -> pd.DataFrame:
    rows = []
    atom_index = 0

    with path.open("r") as handle:
        for line in handle:
            if not line.startswith(("ATOM", "HETATM")):
                continue

            fields = line.split()

            try:
                x = float(fields[-5])
                y = float(fields[-4])
                z = float(fields[-3])
                charge = float(fields[-2])
                radius = float(fields[-1])

                rows.append(
                    (atom_index, x, y, z, charge, radius)
                )
                atom_index += 1

            except (ValueError, IndexError):
                continue

    df = pd.DataFrame(
        rows,
        columns=[
            "file_index", "x", "y", "z",
            "charge", "radius"
        ],
    )

    if df.empty:
        raise ValueError(f"No coordinates found in PQR:\n{path}")

    return df


def load_geometry_shell(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Geometry shell not found:\n{path}")

    suffix = path.suffix.lower()

    if suffix == ".pdb":
        df = read_pdb(path)
        out = df[["file_index", "x", "y", "z"]].copy()

    elif suffix == ".csv":
        df = pd.read_csv(path)

        if {"x", "y", "z"}.issubset(df.columns):
            out = df[["x", "y", "z"]].copy()
        else:
            df = pd.read_csv(path, header=None)
            if df.shape[1] < 3:
                raise ValueError("CSV must contain at least x, y, z.")
            out = df.iloc[:, :3].copy()
            out.columns = ["x", "y", "z"]

        out.insert(0, "file_index", np.arange(len(out)))

    else:
        raise ValueError("Geometry shell must be PDB or CSV.")

    out = out.reset_index(drop=True)
    out.insert(0, "point_id", np.arange(len(out), dtype=int))

    return out


def load_charge_shell(path: Path, source: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Charged shell not found:\n{path}")

    source = source.lower()

    if source in {"bfactor", "occupancy"}:
        df = read_pdb(path)

        out = df[["file_index", "x", "y", "z"]].copy()
        out["charge_raw"] = df[source].to_numpy(dtype=float)

    elif source == "pqr":
        df = read_pqr(path)

        out = df[["file_index", "x", "y", "z"]].copy()
        out["charge_raw"] = df["charge"].to_numpy(dtype=float)

    elif source == "csv":
        df = pd.read_csv(path)

        required = {"x", "y", "z", "charge"}
        if not required.issubset(df.columns):
            raise ValueError(
                "Charged CSV must contain columns x, y, z, charge."
            )

        out = df[["x", "y", "z", "charge"]].copy()
        out = out.rename(columns={"charge": "charge_raw"})
        out.insert(0, "file_index", np.arange(len(out)))

    else:
        raise ValueError(
            "CHARGE_SOURCE must be: bfactor, occupancy, pqr, csv"
        )

    return out.reset_index(drop=True)


print("File readers ready.")


File readers ready.



## 4. Funzioni geometriche

### Orientation plane
Il piano iniziale è il best-fit dei 5 punti di riferimento.

### Direzione della normale
Il segno della normale di un best-fit plane è arbitrario. Qui la rendiamo deterministica: la maggior parte dello shell deve stare **dietro** il piano, quindi `+normal` viene interpretato come verso esterno.

### Contact plane
Il piano viene traslato parallelamente verso il punto più esterno nella direzione `+normal`. Quel piano definisce `depth = 0 Å`.

La depth cresce verso l'interno del nylon.


In [4]:

def unit(v):
    v = np.asarray(v, dtype=float)
    norm = np.linalg.norm(v)
    if norm == 0:
        raise ValueError("Zero-length vector.")
    return v / norm


def plane_from_reference_points(points: np.ndarray) -> Plane:
    if len(points) < 3:
        raise ValueError("At least 3 points are required.")
    return Plane.best_fit(points)


def orient_plane_outward(plane: Plane, shell_xyz: np.ndarray) -> Plane:
    p0 = np.asarray(plane.point, dtype=float)
    n = unit(plane.normal)

    signed = (shell_xyz - p0) @ n

    # Bulk of nylon should be behind the front plane.
    if np.median(signed) > 0:
        n = -n

    return Plane(point=p0, normal=n)


def optional_refit_orientation(
    initial_plane: Plane,
    shell_xyz: np.ndarray,
    enabled: bool,
    band_A: float,
):
    if not enabled:
        return initial_plane, None

    p0 = np.asarray(initial_plane.point, dtype=float)
    n = unit(initial_plane.normal)

    signed = (shell_xyz - p0) @ n
    mask = np.abs(signed) <= band_A

    points = shell_xyz[mask]

    if len(points) < 3:
        raise ValueError(
            "Not enough points for orientation refit. "
            "Increase REFIT_BAND_A."
        )

    refit = Plane.best_fit(points)
    refit = orient_plane_outward(refit, shell_xyz)

    return refit, mask


def build_contact_plane(
    orientation_plane: Plane,
    shell_xyz: np.ndarray,
    quantile: float = 1.0,
):
    if not (0 < quantile <= 1):
        raise ValueError("quantile must be in (0, 1].")

    p0 = np.asarray(orientation_plane.point, dtype=float)
    n = unit(orientation_plane.normal)

    signed = (shell_xyz - p0) @ n

    shift = float(
        np.max(signed) if quantile == 1.0
        else np.quantile(signed, quantile)
    )

    contact_point = p0 + shift * n
    contact_plane = Plane(point=contact_point, normal=n)

    return contact_plane, signed, shift


def contact_signed_distance(contact_plane: Plane, xyz: np.ndarray):
    p0 = np.asarray(contact_plane.point, dtype=float)
    n = unit(contact_plane.normal)
    return (xyz - p0) @ n


def depth_from_contact_plane(contact_plane: Plane, xyz: np.ndarray):
    signed_contact = contact_signed_distance(contact_plane, xyz)
    depth = -signed_contact
    return depth, signed_contact


def deterministic_plane_basis(
    plane: Plane,
    preferred_u_axis: np.ndarray,
):
    n = unit(plane.normal)
    preferred = unit(preferred_u_axis)

    u_raw = preferred - np.dot(preferred, n) * n

    if np.linalg.norm(u_raw) < 1e-8:
        fallback = np.array([0.0, 1.0, 0.0])
        u_raw = fallback - np.dot(fallback, n) * n

    u_axis = unit(u_raw)
    v_axis = unit(np.cross(n, u_axis))

    return u_axis, v_axis


def orthogonal_projection_xyz(plane: Plane, xyz: np.ndarray):
    p0 = np.asarray(plane.point, dtype=float)
    n = unit(plane.normal)

    signed = (xyz - p0) @ n
    projected = xyz - signed[:, None] * n[None, :]

    return projected


def local_coordinates_raw(
    plane: Plane,
    projected_xyz: np.ndarray,
    u_axis: np.ndarray,
    v_axis: np.ndarray,
):
    p0 = np.asarray(plane.point, dtype=float)
    rel = projected_xyz - p0

    u_raw = rel @ u_axis
    v_raw = rel @ v_axis

    return u_raw, v_raw


def geometric_rectangle_center_uv(u_raw, v_raw):
    u_center = 0.5 * (np.min(u_raw) + np.max(u_raw))
    v_center = 0.5 * (np.min(v_raw) + np.max(v_raw))
    return float(u_center), float(v_center)


print("Geometry functions ready.")


Geometry functions ready.



## 5. Caricamento dello shell geometrico


In [5]:

shell_df = load_geometry_shell(NEUTRAL_SHELL_PATH)
shell_xyz = shell_df[["x", "y", "z"]].to_numpy(dtype=float)

print("Shell points:", len(shell_df))
display(shell_df.head())


Shell points: 26638


,point_id,file_index,x,y,z
0,0,0,-3.827,16.363,36.591
1,1,1,-3.596,15.683,37.554
2,2,2,-3.612,16.251,36.962
3,3,3,-3.649,15.361,37.344
4,4,4,-3.883,16.024,36.370



## 6. Orientation plane e controllo dei 5 punti

Verifichiamo quanto i 5 punti di riferimento sono coerenti con il piano che definiscono.


In [6]:

initial_plane = plane_from_reference_points(REFERENCE_POINTS)
initial_plane = orient_plane_outward(initial_plane, shell_xyz)

ref_p0 = np.asarray(initial_plane.point, dtype=float)
ref_n = unit(initial_plane.normal)

reference_signed = (REFERENCE_POINTS - ref_p0) @ ref_n
reference_abs = np.abs(reference_signed)

print("Orientation plane point :", ref_p0)
print("Outward normal          :", ref_n)
print()
print("Reference-point residuals (Å):")
print(reference_abs)
print()
print("RMS residual (Å):",
      np.sqrt(np.mean(reference_abs**2)))
print("Max residual (Å):",
      np.max(reference_abs))


Orientation plane point : [15.2686 33.3366 49.537 ]
Outward normal          : [-0.5106372  -0.01654231  0.85963713]

Reference-point residuals (Å):
[0.09085631 0.31379917 0.11298658 0.04679345 0.06316283]

RMS residual (Å): 0.15853701655451774
Max residual (Å): 0.31379917384562805



## 7. Optional plane refitting

The original five reference points are retained as the default definition of the nylon face. An optional refitting step has been included for future validation, but it is disabled by default because automatically selecting additional face points would introduce another geometric assumption. The stability of the five-point plane is evaluated before deciding whether such a refit is necessary.

For the present nylon structure, the five-point plane is already very stable: the RMS residual is approximately 0.16 Å, the maximum residual is approximately 0.31 Å, and leave-one-out refitting changes the plane normal by less than approximately 0.37°. Therefore, no additional refitting is currently required.


Di default rimane disattivato. Se viene attivato, il notebook usa i punti dello shell entro `REFIT_BAND_A` dal piano iniziale e rifà il best-fit.


In [7]:

orientation_plane, refit_mask = optional_refit_orientation(
    initial_plane,
    shell_xyz,
    enabled=REFIT_ORIENTATION,
    band_A=REFIT_BAND_A,
)

orientation_n = unit(orientation_plane.normal)

if refit_mask is None:
    print("Orientation refit: OFF")
else:
    old_n = unit(initial_plane.normal)
    cosine = np.clip(np.dot(old_n, orientation_n), -1.0, 1.0)
    angle_deg = np.degrees(np.arccos(cosine))

    print("Orientation refit: ON")
    print("Points used:", int(refit_mask.sum()))
    print("Normal change:", angle_deg, "degrees")

print("Final orientation normal:", orientation_n)


Orientation refit: OFF
Final orientation normal: [-0.5106372  -0.01654231  0.85963713]



## 8. Definition of the first-contact plane

The orientation plane is translated along the outward normal until it reaches the outermost shell point. This translated plane is defined as the contact plane and represents depth = 0. High-percentile signed distances are also reported to verify that the maximum is not determined by a single isolated outlier. This definition is intended to provide a physically interpretable reference for the subsequent depth map.

In the current dataset, the maximum lies only about 0.21 Å beyond the 99.9th percentile, indicating that the first-contact position is not dominated by a large isolated outlier.


Il contact plane viene posto al punto più esterno lungo la normale.

Per controllare che il massimo non sia dominato da un singolo outlier, vengono mostrati anche i percentili 99.5 e 99.9.


In [8]:

contact_plane, orientation_signed, contact_shift = build_contact_plane(
    orientation_plane,
    shell_xyz,
    quantile=CONTACT_QUANTILE,
)

q995 = np.quantile(orientation_signed, 0.995)
q999 = np.quantile(orientation_signed, 0.999)
q100 = np.max(orientation_signed)

print("Contact shift from orientation plane (Å):", contact_shift)
print()
print("Upper signed-distance diagnostics:")
print("99.5 percentile:", q995, "Å")
print("99.9 percentile:", q999, "Å")
print("maximum        :", q100, "Å")
print()
print("max - 99.9 percentile:", q100 - q999, "Å")
print("max - 99.5 percentile:", q100 - q995, "Å")
print()
print("Contact plane point:", np.asarray(contact_plane.point))
print("Contact plane normal:", unit(contact_plane.normal))


Contact shift from orientation plane (Å): 0.9017358831833802

Upper signed-distance diagnostics:
99.5 percentile: 0.414064518448697 Å
99.9 percentile: 0.6954227338681854 Å
maximum        : 0.9017358831833802 Å

max - 99.9 percentile: 0.20631314931519473 Å
max - 99.5 percentile: 0.48767136473468314 Å

Contact plane point: [14.80814011 33.32168321 50.31216565]
Contact plane normal: [-0.5106372  -0.01654231  0.85963713]



## 9. Depthmap (Signed depth instead of absolute distance)

The original script selected points using the absolute distance from the plane. In this version, the signed distance is retained so that the side of the plane is not lost. With the normal pointing outward, points inside the nylon have a negative signed distance and their depth is defined as depth = -signed_distance. The front surface is therefore selected using 0 ≤ depth ≤ FACE_DEPTH_CUTOFF_A. This same depth value can later be used directly in the SURFMATCH distance-dependent scoring.


Con la normale rivolta verso l'esterno:

- `signed_contact_distance = 0` sul piano di primo contatto;
- i punti interni hanno signed distance negativa;
- definiamo:

\[
depth = -signed\_contact\_distance
\]

e selezioniamo:

\[
0 \le depth \le FACE\_DEPTH\_CUTOFF
\]

Questo è intenzionalmente diverso dall'usare la distanza assoluta.


In [9]:

all_depth, all_signed_contact = depth_from_contact_plane(
    contact_plane,
    shell_xyz,
)

EPS = 1e-8

front_mask = (
    (all_depth >= -EPS)
    & (all_depth <= FACE_DEPTH_CUTOFF_A + EPS)
)

front_df = shell_df.loc[front_mask].copy()
front_df["signed_contact_distance_A"] = all_signed_contact[front_mask]
front_df["depth_A"] = all_depth[front_mask]

print("Total shell points :", len(shell_df))
print("Selected front slab:", len(front_df))
print()
print("Depth min :", front_df["depth_A"].min(), "Å")
print("Depth max :", front_df["depth_A"].max(), "Å")
print("Depth mean:", front_df["depth_A"].mean(), "Å")

positive_signed = np.sum(
    front_df["signed_contact_distance_A"] > 1e-6
)

print()
print(
    "Selected points unexpectedly outside contact plane:",
    int(positive_signed)
)


Total shell points : 26638
Selected front slab: 6148

Depth min : -8.881784197001252e-16 Å
Depth max : 7.99972621437515 Å
Depth mean: 2.9002711595038675 Å

Selected points unexpectedly outside contact plane: 0



## 10. Proiezione ortogonale e coordinate metriche `u,v`


Physical planar coordinates and a reproducible origin

Selected shell points are orthogonally projected onto the contact plane and expressed in an orthonormal local coordinate system. Therefore, the resulting u and v coordinates retain physical units in Å. The origin is no longer defined by the first point in the file. Instead, (0,0) is placed at the geometric center of the projected rectangular surface, making the representation independent of file ordering and easier to reproduce across calculations.


Deterministic in-plane axes
The local planar axes are explicitly constructed as orthonormal vectors. A preferred global direction is projected onto the plane to define u, and v is obtained from the cross product with the surface normal. This avoids arbitrary rotations of the 2D map between different executions and preserves metric distances on the plane.



Gli stessi punti vengono proiettati ortogonalmente sul contact plane.

Gli assi locali sono ortonormali, quindi `u` e `v` sono in **Å**.

L'origine non è più il primo punto del file. Viene definita come il centro geometrico del rettangolo proiettato:

\[
u_c = \frac{u_{min}+u_{max}}{2}
\]

\[
v_c = \frac{v_{min}+v_{max}}{2}
\]

Dopo il ricentraggio, il centro della faccia è `(0,0)`.


In [10]:

front_xyz = front_df[["x", "y", "z"]].to_numpy(dtype=float)

projected_xyz = orthogonal_projection_xyz(
    contact_plane,
    front_xyz,
)

u_axis, v_axis = deterministic_plane_basis(
    contact_plane,
    PREFERRED_U_AXIS,
)

u_raw, v_raw = local_coordinates_raw(
    contact_plane,
    projected_xyz,
    u_axis,
    v_axis,
)

u_center_raw, v_center_raw = geometric_rectangle_center_uv(
    u_raw,
    v_raw,
)

u = u_raw - u_center_raw
v = v_raw - v_center_raw

front_df["proj_x"] = projected_xyz[:, 0]
front_df["proj_y"] = projected_xyz[:, 1]
front_df["proj_z"] = projected_xyz[:, 2]

front_df["u_A"] = u
front_df["v_A"] = v
front_df["radial_distance_A"] = np.sqrt(u**2 + v**2)

print("u axis:", u_axis)
print("v axis:", v_axis)
print()
print("u range:", front_df["u_A"].min(),
      "to", front_df["u_A"].max(), "Å")
print("v range:", front_df["v_A"].min(),
      "to", front_df["v_A"].max(), "Å")
print()
print("Rectangle center after recentering:")
print(
    0.5 * (front_df["u_A"].min() + front_df["u_A"].max()),
    0.5 * (front_df["v_A"].min() + front_df["v_A"].max())
)


u axis: [ 0.85979628 -0.00982456  0.51054268]
v axis: [-1.73472348e-18  9.99814898e-01  1.92397966e-02]

u range: -24.09942945715506 to 24.09942945715506 Å
v range: -35.930700547418866 to 35.930700547418866 Å

Rectangle center after recentering:
0.0 0.0



## 11. Geometric validation of the orthogonal projection

An orthogonal projection cannot increase the Euclidean distance between the same pair of points. As an internal validation, the nearest neighbour of each selected point is identified in 3D and the distance of the same pair is measured after projection. Any case in which the projected distance exceeds the original 3D distance would indicate an implementation error.



Per ogni punto:

1. troviamo il nearest neighbour in 3D;
2. manteniamo **la stessa coppia di punti**;
3. misuriamo la loro distanza dopo la proiezione 2D;
4. verifichiamo:

\[
d_{2D} \le d_{3D}
\]

Se compaiono violazioni numeriche reali, c'è un problema nell'implementazione.


In [11]:

tree3d = cKDTree(front_xyz)

dist3d, idx3d = tree3d.query(front_xyz, k=2)

nn_dist_3d = dist3d[:, 1]
nn_idx = idx3d[:, 1]

uv = front_df[["u_A", "v_A"]].to_numpy(dtype=float)

nn_uv_delta = uv - uv[nn_idx]
same_pair_dist_2d = np.linalg.norm(nn_uv_delta, axis=1)

violations = same_pair_dist_2d > (nn_dist_3d + 1e-8)

print("Pairs checked:", len(front_df))
print("Projection-distance violations:", int(violations.sum()))
print()
print("Median same-pair distance in 3D:",
      np.median(nn_dist_3d), "Å")
print("Median same-pair distance after projection:",
      np.median(same_pair_dist_2d), "Å")
print()
print("Maximum d2D / d3D ratio:",
      np.max(same_pair_dist_2d / nn_dist_3d))


Pairs checked: 6148
Projection-distance violations: 0

Median same-pair distance in 3D: 0.3023478002104327 Å
Median same-pair distance after projection: 0.2235961260581646 Å

Maximum d2D / d3D ratio: 0.9999945884046276


## Output
No distance-increase violations were detected in the present dataset. Therefore, the apparent gaps and periodic spacing visible in the planar map are not introduced by the projection algorithm, but originate from the discrete structure of the nylon shell.


## 12. Controllo 3D

- shell completo;
- spessore frontale selezionato, colorato per depth;
- contact plane.


In [12]:

def plane_patch(
    plane,
    u_axis,
    v_axis,
    half_size=45.0,
    resolution=25,
):
    origin = np.asarray(plane.point, dtype=float)

    grid = np.linspace(-half_size, half_size, resolution)
    U, V = np.meshgrid(grid, grid)

    xyz = (
        origin[None, None, :]
        + U[:, :, None] * u_axis[None, None, :]
        + V[:, :, None] * v_axis[None, None, :]
    )

    return xyz[..., 0], xyz[..., 1], xyz[..., 2]


plane_x, plane_y, plane_z = plane_patch(
    contact_plane,
    u_axis,
    v_axis,
    half_size=PLANE_HALF_SIZE_A,
)

fig3d = go.Figure()

fig3d.add_trace(
    go.Scatter3d(
        x=shell_df["x"],
        y=shell_df["y"],
        z=shell_df["z"],
        mode="markers",
        marker=dict(size=1.5, opacity=0.25),
        name="Full shell",
    )
)

fig3d.add_trace(
    go.Scatter3d(
        x=front_df["x"],
        y=front_df["y"],
        z=front_df["z"],
        mode="markers",
        marker=dict(
            size=3,
            color=front_df["depth_A"],
            colorscale="Viridis",
            colorbar=dict(title="Depth (Å)"),
        ),
        name="Selected front slab",
    )
)

fig3d.add_trace(
    go.Surface(
        x=plane_x,
        y=plane_y,
        z=plane_z,
        opacity=0.35,
        showscale=False,
        name="Contact plane",
    )
)

fig3d.update_layout(
    title="Nylon — 3D control of contact plane and selected front slab",
    scene_aspectmode="data",
)

fig3d.show()



# MAPPA 1 — DEPTH

## 13. Proiezione 2D colorata per depth

Questa **non è una heatmap interpolata**.

Ogni marker corrisponde a un punto reale dello shell. Le coordinate sono le dimensioni fisiche della proiezione sul piano in Å.


In [13]:

fig_depth = go.Figure(
    go.Scattergl(
        x=front_df["u_A"],
        y=front_df["v_A"],
        mode="markers",
        marker=dict(
            size=MARKER_SIZE,
            color=front_df["depth_A"],
            colorscale="Viridis",
            colorbar=dict(title="Depth (Å)"),
            line=dict(width=0),
        ),
        customdata=np.column_stack([
            front_df["point_id"],
            front_df["x"],
            front_df["y"],
            front_df["z"],
            front_df["depth_A"],
            front_df["signed_contact_distance_A"],
        ]),
        hovertemplate=(
            "point_id = %{customdata[0]:.0f}<br>"
            "u = %{x:.3f} Å<br>"
            "v = %{y:.3f} Å<br>"
            "x = %{customdata[1]:.3f} Å<br>"
            "y = %{customdata[2]:.3f} Å<br>"
            "z = %{customdata[3]:.3f} Å<br>"
            "depth = %{customdata[4]:.4f} Å<br>"
            "signed contact distance = %{customdata[5]:.4f} Å"
            "<extra></extra>"
        ),
    )
)

fig_depth.update_layout(
    title="Nylon planar projection — depth map",
    template="plotly_white",
    width=900,
    height=900,
    xaxis=dict(
        title="u (Å)",
        scaleanchor="y",
        scaleratio=1,
        zeroline=True,
    ),
    yaxis=dict(
        title="v (Å)",
        zeroline=True,
    ),
)

fig_depth.show()



# MAPPA 2 — CARICA

## 14. Associazione delle cariche - One-to-one mapping of charge and geometry

Charges are not projected independently. Instead, they are assigned to the already projected geometric points. The code first checks whether geometry and charge files contain the same coordinates in the same order; otherwise, a coordinate-based nearest-neighbour match is attempted within a strict tolerance. Non-unique or geometrically inconsistent matches are rejected. This ensures that depth and charge always refer to the same physical shell point.

**For the present files, the geometry and charge shells match exactly in the same order, with a maximum coordinate mismatch of 0.0 Å.**

La carica viene trasferita agli stessi `point_id` geometrici.

Se i file non sono nello stesso ordine, il notebook usa un matching per coordinate e rifiuta associazioni oltre la tolleranza o non biunivoche.


In [14]:

def attach_charges_to_geometry(
    geometry_df: pd.DataFrame,
    charge_df: pd.DataFrame,
    tolerance_A: float,
):
    geom_xyz = geometry_df[["x", "y", "z"]].to_numpy(dtype=float)
    charge_xyz = charge_df[["x", "y", "z"]].to_numpy(dtype=float)

    if len(geometry_df) == len(charge_df):
        same_order_dist = np.linalg.norm(
            geom_xyz - charge_xyz,
            axis=1,
        )

        if np.max(same_order_dist) <= tolerance_A:
            out = geometry_df.copy()
            out["charge_raw"] = charge_df["charge_raw"].to_numpy(dtype=float)

            return out, {
                "method": "same_order",
                "max_match_distance_A": float(np.max(same_order_dist)),
            }

    tree = cKDTree(charge_xyz)
    distances, indices = tree.query(geom_xyz, k=1)

    if np.max(distances) > tolerance_A:
        raise ValueError(
            "Charged shell does not geometrically match the neutral shell. "
            f"Maximum nearest-point distance = {np.max(distances):.6f} Å, "
            f"tolerance = {tolerance_A:.6f} Å."
        )

    if len(np.unique(indices)) != len(indices):
        raise ValueError(
            "Charge matching is not one-to-one: at least one charged point "
            "would be assigned to multiple geometry points."
        )

    out = geometry_df.copy()
    out["charge_raw"] = (
        charge_df.iloc[indices]["charge_raw"].to_numpy(dtype=float)
    )

    return out, {
        "method": "nearest_coordinate",
        "max_match_distance_A": float(np.max(distances)),
    }


if CHARGED_SHELL_PATH is None:
    charged_front_df = None
    charge_match_info = None

    print(
        "CHARGED_SHELL_PATH is None.\n"
        "Insert the charged-shell path in the settings cell "
        "and rerun from this cell."
    )

else:
    charge_shell_df = load_charge_shell(
        Path(CHARGED_SHELL_PATH),
        CHARGE_SOURCE,
    )

    geometry_with_charge, charge_match_info = attach_charges_to_geometry(
        shell_df,
        charge_shell_df,
        tolerance_A=COORD_MATCH_TOL_A,
    )

    charge_lookup = (
        geometry_with_charge
        .set_index("point_id")["charge_raw"]
    )

    charged_front_df = front_df.copy()
    charged_front_df["charge_raw"] = (
        charged_front_df["point_id"].map(charge_lookup)
    )

    if charged_front_df["charge_raw"].isna().any():
        raise ValueError("Some selected points have no matched charge.")

    max_abs_charge = float(
        np.nanmax(np.abs(charged_front_df["charge_raw"]))
    )

    if max_abs_charge > 0:
        charged_front_df["charge_normalized"] = (
            charged_front_df["charge_raw"] / max_abs_charge
        )
    else:
        charged_front_df["charge_normalized"] = 0.0

    print("Charge matching:", charge_match_info)
    print()
    print("Raw charge:")
    print(charged_front_df["charge_raw"].describe())
    print()
    print("Normalized charge:")
    print(charged_front_df["charge_normalized"].describe())


Charge matching: {'method': 'same_order', 'max_match_distance_A': 0.0}

Raw charge:
count    6148.000000
mean       -0.008448
std         0.296380
min        -0.650000
25%        -0.030000
50%         0.040000
75%         0.100000
max         0.780000
Name: charge_raw, dtype: float64

Normalized charge:
count    6148.000000
mean       -0.010831
std         0.379975
min        -0.833333
25%        -0.038462
50%         0.051282
75%         0.128205
max         1.000000
Name: charge_normalized, dtype: float64



## 15. Proiezione 2D colorata per carica

Raw versus normalized charge values

The original charge value is always retained as charge_raw. A normalized value is additionally generated for visualization, but it should not be interpreted as a new physical charge. This distinction is important because the exact normalization procedure used by SURFMAP must be reproduced before a quantitative comparison between the planar map and the SURFMAP map is performed.

Usa **esattamente le stesse coordinate `u,v` e gli stessi punti** della depthmap.

Non viene eseguita una seconda proiezione geometrica.

La figura usa la carica normalizzata per la visualizzazione, ma il valore grezzo resta nel dataset.


In [15]:

CHARGE_COLORSCALE = [
    [0.00, "rgb(230,0,0)"],
    [0.25, "rgb(255,160,160)"],
    [0.50, "rgb(255,255,255)"],
    [0.75, "rgb(160,160,255)"],
    [1.00, "rgb(0,0,230)"],
]

if charged_front_df is None:
    print("Charged shell not loaded yet.")

else:
    fig_charge = go.Figure(
        go.Scattergl(
            x=charged_front_df["u_A"],
            y=charged_front_df["v_A"],
            mode="markers",
            marker=dict(
                size=MARKER_SIZE,
                color=charged_front_df["charge_normalized"],
                colorscale=CHARGE_COLORSCALE,
                cmin=-1,
                cmax=1,
                colorbar=dict(
                    title="Normalized charge",
                    tickvals=[
                        -1, -0.67, -0.33,
                        0, 0.33, 0.67, 1
                    ],
                ),
                line=dict(width=0),
            ),
            customdata=np.column_stack([
                charged_front_df["point_id"],
                charged_front_df["charge_raw"],
                charged_front_df["charge_normalized"],
                charged_front_df["depth_A"],
            ]),
            hovertemplate=(
                "point_id = %{customdata[0]:.0f}<br>"
                "u = %{x:.3f} Å<br>"
                "v = %{y:.3f} Å<br>"
                "charge raw = %{customdata[1]:.6f}<br>"
                "charge normalized = %{customdata[2]:.6f}<br>"
                "depth = %{customdata[3]:.4f} Å"
                "<extra></extra>"
            ),
        )
    )

    fig_charge.update_layout(
        title="Nylon planar projection — charge map",
        template="plotly_white",
        width=900,
        height=900,
        xaxis=dict(
            title="u (Å)",
            scaleanchor="y",
            scaleratio=1,
            zeroline=True,
        ),
        yaxis=dict(
            title="v (Å)",
            zeroline=True,
        ),
    )

    fig_charge.show()



## 16. Dataset finale

Questa tabella è l'output numerico principale.

Nessuna ROI è applicata.


In [16]:

base_columns = [
    "point_id",
    "x", "y", "z",
    "proj_x", "proj_y", "proj_z",
    "u_A", "v_A",
    "radial_distance_A",
    "signed_contact_distance_A",
    "depth_A",
]

if charged_front_df is None:
    final_projection_df = front_df[base_columns].copy()
else:
    final_projection_df = charged_front_df[
        base_columns + ["charge_raw", "charge_normalized"]
    ].copy()

display(final_projection_df.head(10))

print("Rows:", len(final_projection_df))
print("Columns:", list(final_projection_df.columns))


,point_id,x,y,z,proj_x,proj_y,proj_z,u_A,v_A,radial_distance_A,signed_contact_distance_A,depth_A,charge_raw,charge_normalized
0,0,-3.827,16.363,36.591,-4.847709,16.329934,38.309322,-20.375985,-17.532701,26.880780,-1.998892,1.998892,-0.60,-0.769231
1,1,-3.596,15.683,37.554,-4.248477,15.661863,38.652418,-19.679039,-18.194047,26.800894,-1.277770,1.277770,0.06,0.076923
2,2,-3.612,16.251,36.962,-4.524969,16.221424,38.498946,-20.000617,-17.637542,26.666600,-1.787901,1.787901,-0.60,-0.769231
3,3,-3.649,15.361,37.344,-4.377119,15.337412,38.569759,-19.828658,-18.520028,27.132400,-1.425903,1.425903,0.06,0.076923
4,4,-3.883,16.024,36.370,-4.983254,15.988357,38.222233,-20.533633,-17.875890,27.224576,-2.154668,2.154668,-0.60,-0.769231
5,5,-3.623,15.403,37.035,-4.493893,15.374787,38.501113,-19.964474,-18.483981,27.207311,-1.705502,1.705502,0.06,0.076923
6,6,-3.584,14.883,36.490,-4.699904,14.846850,38.368580,-20.204079,-19.014370,27.744388,-2.185318,2.185318,0.06,0.076923
7,7,-3.848,15.559,35.882,-5.167666,15.516249,38.103604,-20.748116,-18.350193,27.698627,-2.584351,2.584351,-0.60,-0.769231
8,8,-3.590,14.390,36.196,-4.829231,14.349855,38.282195,-20.354494,-19.512935,28.196809,-2.426832,2.426832,-0.03,-0.038462
9,9,-3.594,14.375,36.142,-4.855765,14.334125,38.266130,-20.385355,-19.528971,28.230186,-2.470961,2.470961,-0.03,-0.038462


Rows: 6148
Columns: ['point_id', 'x', 'y', 'z', 'proj_x', 'proj_y', 'proj_z', 'u_A', 'v_A', 'radial_distance_A', 'signed_contact_distance_A', 'depth_A', 'charge_raw', 'charge_normalized']



## 17. Salvataggio CSV, metadata JSON e HTML

Reproducibility of the planar representation

The final CSV stores the complete point-level representation required for subsequent SURFMATCH analysis, while the metadata JSON stores the geometric reference system used to generate it. Saving the plane origin, normal, local axes, center, depth cutoff and contact-plane definition makes it possible to reconstruct exactly the same planar projection in later analyses without relying on implicit conventions or file ordering.

Il JSON permette di ricostruire esattamente lo stesso sistema di riferimento.


In [17]:

OUTPUT_DIR = Path.cwd() / "nylon_planar_projection_corrected"
OUTPUT_DIR.mkdir(exist_ok=True)

csv_path = OUTPUT_DIR / "nylon_planar_projection_points.csv"
metadata_path = OUTPUT_DIR / "nylon_planar_projection_metadata.json"
depth_html_path = OUTPUT_DIR / "nylon_depth_map.html"

final_projection_df.to_csv(csv_path, index=False)

orientation_point = np.asarray(orientation_plane.point, dtype=float)
orientation_normal = unit(orientation_plane.normal)
contact_point = np.asarray(contact_plane.point, dtype=float)
contact_normal = unit(contact_plane.normal)

metadata = {
    "neutral_shell_path": str(NEUTRAL_SHELL_PATH),
    "charged_shell_path": (
        None if CHARGED_SHELL_PATH is None
        else str(CHARGED_SHELL_PATH)
    ),
    "charge_source": CHARGE_SOURCE,
    "reference_points": REFERENCE_POINTS.tolist(),
    "refit_orientation": bool(REFIT_ORIENTATION),
    "refit_band_A": float(REFIT_BAND_A),
    "orientation_plane_point": orientation_point.tolist(),
    "orientation_plane_normal_outward": orientation_normal.tolist(),
    "contact_quantile": float(CONTACT_QUANTILE),
    "contact_plane_point": contact_point.tolist(),
    "contact_plane_normal_outward": contact_normal.tolist(),
    "contact_shift_A": float(contact_shift),
    "face_depth_cutoff_A": float(FACE_DEPTH_CUTOFF_A),
    "u_axis": u_axis.tolist(),
    "v_axis": v_axis.tolist(),
    "u_center_before_recentering_A": float(u_center_raw),
    "v_center_before_recentering_A": float(v_center_raw),
    "projected_u_range_A": [
        float(front_df["u_A"].min()),
        float(front_df["u_A"].max()),
    ],
    "projected_v_range_A": [
        float(front_df["v_A"].min()),
        float(front_df["v_A"].max()),
    ],
    "selected_point_count": int(len(front_df)),
    "contact_diagnostics_A": {
        "q99_5": float(q995),
        "q99_9": float(q999),
        "maximum": float(q100),
        "max_minus_q99_9": float(q100 - q999),
        "max_minus_q99_5": float(q100 - q995),
    },
    "charge_matching": charge_match_info,
}

with metadata_path.open("w") as f:
    json.dump(metadata, f, indent=2)

fig_depth.write_html(
    depth_html_path,
    include_plotlyjs="cdn",
    full_html=True,
)

print("Saved:")
print("CSV       :", csv_path)
print("Metadata  :", metadata_path)
print("Depth HTML:", depth_html_path)

if charged_front_df is not None:
    charge_html_path = OUTPUT_DIR / "nylon_charge_map.html"

    fig_charge.write_html(
        charge_html_path,
        include_plotlyjs="cdn",
        full_html=True,
    )

    print("Charge HTML:", charge_html_path)


Saved:
CSV       : /home/chiara/Downloads/nylon_planar_projection_corrected/nylon_planar_projection_points.csv
Metadata  : /home/chiara/Downloads/nylon_planar_projection_corrected/nylon_planar_projection_metadata.json
Depth HTML: /home/chiara/Downloads/nylon_planar_projection_corrected/nylon_depth_map.html
Charge HTML: /home/chiara/Downloads/nylon_planar_projection_corrected/nylon_charge_map.html



# 18. Validation summary

**The revised planar projection was validated numerically on the current nylon shell. The five reference points define a stable plane (RMS residual ≈ 0.16 Å; maximum residual ≈ 0.31 Å), and the contact-plane position is not dominated by a strong isolated outlier. The local u,v axes are orthonormal and the projected coordinates reproduce the analytical orthogonal projection to numerical precision (~10⁻¹⁴ Å). No pairwise distance-increase violations were detected, confirming that the visible spacing between projected points is not generated by the projection algorithm. Geometry and B-factor charge values also match exactly point-by-point. The resulting planar map therefore appears geometrically valid and suitable as a reference representation of the nylon surface for the subsequent comparison with SURFMAP. The main remaining parameter to validate is the 8 Å front-surface depth cutoff, because the deepest selected points are concentrated mainly near the perimeter of the rectangle. This does not affect the planned central-region validation, but it should be reconsidered if the complete projected surface is later used directly for SURFMATCH scoring. Charge normalization should also be treated as visualization-only until the exact SURFMAP normalization procedure is reproduced.**
